# STA 141a Final Project: Shreya Maddhali

## 0. Abstract

Wildfires are a natural disaster that impacts many parts of the world, especially in the United States, where large fires cause significant environmental and economic damage each year. In this project, I develop a predictive model using historical wildfire records. The dataset is transformed into a grid-by-month panel. The model estimates the probability that a wildfire will occur during a specific month in a specific location.

The analysis first explores overall patterns across the United States using exploratory figures to highlight temporal trends. The additional model that is included focuses on California specifically, due to its frequent and severe wildfire activity and my personal experience with wildfires in this region. A binary variable is the outcome variable, indicating whether a fire occurs in a given grid cell during a given month. Several predictors are created, including geographic location, recent fire activity, and seasonal trends.

Two models are compared in this project and they will be evaluated along with their respective AUC metrics. The first is a logistic regression model,  which serves as the baseline, the second is XGBoost, and then the final one is a more flexible neural network model. Model performance is evaluated using time-aware cross-validation and the area under the ROC curve. The results highlight strong temporal patterns in wildfire activity and show that seasonality and past fire activity are important predictors, which improves the predictive performance of the model.

#### Figure 0. Abstract Layout of Report: Spatial-Temporal Wildfire Risk Prediction Pipeline: Made by Shreya Maddhali

<figure>
<img src="AB.png" width="1000">
</figure>

## 1. Introduction

Wildfires are a devastating natural disaster that burns about 1.5 million acres each year in the United States. Over the last ten years, the U.S. Department of Agriculture published that 54% of wildfires are caused by human activity, while the remaining 46% are ignited by natural causes (i.e. lightning). As of 2025, this statistic has increased to a whopping 85% of human activity-related wildfires. 

I take a special interest in wildfires because I come from Southern California, where wildfires have been incredibly dangerous and catastrophic. Specifically, the Woolsey Fire threatened and forced my family and me to evacuate back in 2018, because the fire began 2 miles away from my house. This fire burned 96,949 acres, destroyed 1,643 structures, killed three people, and prompted the evacuation of more than 295,000 people (https://en.wikipedia.org/wiki/Woolsey_Fire).

In the general Ventura County area (where I am from and where the Woolsey fire burned), there are tons of prevention steps that are taken prior to the windy season. There are goats that are organized to eat the dry brush and mitigate the risk of fire. Local communities have begun a wildfire house evalutation and risk assessment to create fire-resistant landscaping.

#### Figure 1. Early Days of the Woolsey Fire in Southern California, 2018: Taken by Shreya Maddhali

<figure>
<img src="wildfirewool.png" width="900">
</figure>

## 2. Data Preprocessing

This project uses the 1.88 Million US Wildfires dataset from Kaggle, which contains 24 years of geo-referenced wildfire records across the United States. The dataset includes variables such as latitude, longitude, discovery date, and fire size. This dataset is very large, which is why I took a subset of this to narrow it down to about 100k fires as a preprocessing step. A notable wildfire was defined to be greater than or equal to 300 acres in order to focus on fires that had a greater ecological impact while being significant enough for our model to predict. Out of all of the records (100,000) from our dataset, I found that there were 1,381 records that met this threshold.

First, I aggregated into a space–time grid to turn the problem into a binary classification one. Latitude and longitude were grouped to make these geographic grid cells, and time was encoded into monthly units using the variable ym, which represents year–month combinations. The dataset was then summarized to count the number of fires in each grid cell for each month. This produced the dataset grid_month_fire, which contains the following variables:

- lat_bin
- lon_bin
- ym
- n_fires
- fire_this_month

A fire that occcured is indicated by the variable fire_this_month = 1, otherwise, the binary indicator variable would be set to 0.

 After this initial grid_month_fire was made, a full panel was created by using all of the combinations of months and grid cells. Any missing values were imputed with zeros to make sure that the dataset included fires and non-fires.

 
From this panel, the final modeling dataset model_df was constructed. The response variable y_next indicates whether a fire occurs in the next month, highlighting the predictive value of the model. At this stage, additional predictors were created:
- spatial location (lat_bin, lon_bin)
- seasonal variables based on month
- rolling counts of past fires
- time index variables

These predictors were chosen based on the exploratory data analysis (see below).

The analysis is first performed using the full United States dataset, and then as a further look at the data, a random forest analysis is performed on the California dataset because of the simplicity and robustness of the model.

#### Table 1: Summary of the final panel dataset used for wildfire prediction

| Property | Value |
|----------|--------|
| Number of rows | 147,576 |
| Year range | 1992–2015 |
| Spatial unit | 1° × 1° lat–lon grid (floor bins) |
| Temporal unit | Monthly |
| Outcome variable | y_next = 1 if any notable fire occurs in grid cell next month |
| Predictors | lat_bin, lon_bin, state, month_cos, month_sin, year, n_fires, fire_lag12, fire_roll3, fire_roll12, size_roll3, size_roll12 |

The dataset contains 147,576 grid–month observations covering 1992–2015. This dataset was constructed using a 1° × 1° latitude–longitude grid with monthly time resolution.  

The outcome variable indicates whether a notable wildfire occurs in the following month, and predictors include spatial location, seasonal terms, year, and rolling fire history variables.

#### Table 2: Outcome distribution in the final grid–month panel dataset

| Property | Value |
|----------|--------|
| Total observations | 147,576 |
| Positive outcomes (y_next = 1) | 1,311 (0.89%) |
| Negative outcomes (y_next = 0) | 146,265 (99.11%) |

Only 0.89% of observations correspond to a notable wildfire occurring in the following month, showing an imbalance in the prediction set.

## 3. Exploratory Data Analysis

The spatial EDA plot (see figure 2 and 3 below) showed that the mean next_month notable fire rate (denoted as mean y_next in the code) aggregated by 1 degree grid cell across the United States. The map had revealed that there were indicators of strong geographical clustering. Rates of fires seemed to be substantially elevated in the Western United States, including California, Washington, and Oregon. There were also higher fire rates in the Southwestern states, such as Arizona and New Mexico.

Wildfires, based on preliminary domain research, is heavily impacted on spatial factors. Wildfires are deeply tied to climate and overall topography. Location-based features are predicted to have a substantial predictive significance. 

The seasonality plot showed the variable mean y_next against calendar month in order to understand deeper temporal patterns in this data. From the plot, we learned that December through February show the smallest fire probability, while months more in the springtime and summertime are when the plot hits a peak. This distribution looks unimodal with summer at the peak; this could be because summer usually has drier conditions and higher temperatures. 

In order to encode the relationship without any discontinuities, cyclical encoding was used. Month was transformed into the two harmonic features of sine and cosine, where month_sin = sin (2pi * month / 12) and month_cos = cos (2pi * month/12). These features were encoded this way because December, for example, and January, are close in reality, but far numerically.

There were also five lagged features that were constructed in order to capture the autocorrelation in fire activity that is based on temporal factors. 
 - fire_lag12 is defined as the fire count 12 months prior
 - fire_roll3 is defined as the sum over the past three months
 - fire_roll12 is the sum over the past 12 months
 - size_roll3 is the total fire acreage over the past 3 months
 - size_roll12 is defined as the total fire acreage for the past 12 months

The summary statistics show that the median features for the lagged features were all 0, where the maximum values for size_roll3 and size_roll12 reached 344,833 acres. This shows that there were only a small number of grid cells that had experienced fire events that would be noted as "extreme." The heavier-tailed distributions is indicative that lagged features are concentrated in high-risk regions.

Notable wildfire risk exhibits substantial spatial heterogeneity, with certain grid cells emerging as persistent hotspots. This suggests that location-specific baseline risk is an important component of prediction.
Fire occurrence displays strong seasonality, with clear monthly patterns. This motivates the inclusion of cyclical seasonal features (e.g., sine and cosine of month) to capture recurring annual dynamics.

#### Figure 2: Seasonal pattern of the mean probability of a notable wildfire occurring in the next month.

<figure>
<img src="Fig1.png" width="800">
<figcaption>
This plot shows strong geographic variation in wildfire risk across the United States. The higher probabilities are higher in the western region of the country, particularly in Oregon, Washington, and California, where dry climate conditions, vegetation, and seasonal weather patterns contribute to increased fire activity. Contrastingly, much of the central and eastern United States shows very low probabilities, reflecting the sparsity of wildfires that we would consider notable in those regions. These spatial differences suggest that location variables such as latitude, longitude, and state are important.
</figcaption>
</figure>

#### Figure 3: Seasonal pattern of the mean probability of a notable wildfire occurring in the next month.

<figure>
<img src="Fig2.png" width="700">
<figcaption>
The plot shows the average value of y_next by calendar month, aggregated over all grid cells and years in the dataset.

The probability of a notable wildfire is highly seasonal, with the lowest values in winter months and the highest values in late spring and summer. The risk increases sharply beginning in month 5 (May), going up to month 7 (July), and then declines after month 8 (August). The clear seasonal trend supports including month_sin and month_cos in the predictive model to capture cyclic impacts.
</figcaption>
</figure>

## 4. Modeling

The task at hand is a binary classification model. Given all of the observations from a year and before that year, predict whether or not y_next will be equal to 1 for each of the grid-cell-months in the following year. Since the outcome is usually rare for a fire to be a notable one (0.89% from Table 2), AUC-ROC is going to be used as the primary metric in order to measure discriminate abilities across all of the different classification thresholds.

The **baseline** model is a logistic regression model with a binomial link function, since this is an interpretable model. For binomial models, logistic regression is usually the go-to choice because it is incredibly fast to fit. The downside to a logistic regression model for this specific dataset is that high-dimensional data can be oversimplified in this method. The more predictors there are, the more overfitting is likely to happpen. This is why the theme of regularization is common when utilizing these models.

The **second** model is XGBoost, which is also known as eXtreme Gradient Boosting. This technique has parallel and distributed computing, cache-aware prefetching algorithms, built in regularization, and handles missing values (IBM: https://www.ibm.com/think/topics/xgboost).

The model was configured using the following hyperparameters: objective = binary:logistic, eval_metric= AUC, learning rate = 0.1, max_depth = 8, subsample = 0.8, colsample_bytree = 0.8, and nrounds = 200.

XGBoost was picked because of its ability to understand complex and nonlinear relationships between the predictors. Since we observed that there were quite a few graphs with heavier tails in terms of temporal patterns, XGBoost would fit this prediction task well. The same 12 features that were used in the baseline logistic regression model were also used for the XGBoost model.

The **third** model is a single-hidden-layer feedforward neural network, using the R nnet package. The features were reduced for this model to 5 predictors: month_sin, month_cos, lat_bin, lon_bin, and fire_roll12. The network has 8 hidden units and uses a sigmoid activation function in order to accommodate the binary classification of the model. The network architecture also has a L2 weight decay of 1e^-4 and a maximum of 150 training iterations. The feature reduction was necessary for this neural network because of the idea of preserving the most informative signals across all 20 rolling validation folds.

### Modeling Formulas:

#### Logistic Regression

$$\log \frac{P(y_{next}=1)}{1 - P(y_{next}=1)} = \beta_0 + \sum_{j=1}^{12} \beta_j x_j$$

 $x_j$ is each of the 12 predictors: `lat_bin`, `lon_bin`, `month_sin`, `month_cos`, `year`, `n_fires`, `fire_lag12`, `fire_roll3`, `fire_roll12`, `size_roll3`, `size_roll12`, and `state`.

#### XGBoost

XGBoost builds $M = 200$ regression trees. 

$$\hat{y}_i^{(m)} = \hat{y}_i^{(m-1)} + \eta \cdot f_m(\mathbf{x}_i)$$

$$\mathcal{L} = \sum_{i=1}^{n} \ell\!\left(y_i, \hat{y}_i\right) + \sum_{m=1}^{M} \Omega(f_m)$$

 $\ell$ $\Omega(f_m) = \gamma T + \frac{1}{2}\lambda \|\mathbf{w}\|^2$ 

#### Neural Network

A single hidden layer feedforward network with sigmoid activations:

$$\mathbf{h} = \sigma(W_1 \mathbf{x} + \mathbf{b}_1), \qquad \hat{y} = \sigma(\mathbf{w}_2^\top \mathbf{h} + b_2)$$

$\mathbf{x} \in \mathbb{R}^5$, $\mathbf{h} \in \mathbb{R}^8$, $\sigma(z) = \frac{1}{1+e^{-z}}$.  $\lambda = 10^{-4}$:

$$\mathcal{L} = -\sum_i \left[ y_i \log \hat{y}_i + (1-y_i)\log(1-\hat{y}_i) \right] + \lambda \|W\|_F^2$$

## 5. Model Evaluation and Validation

The goal of the model is to predict whether a wildfire will occur in a given grid cell during the next month. This is treated as a binary classification problem.

Prior to modeling, in order to prevent any data leakage and to highlight temporal structures of the panel, a rolling cross-validation scheme was used, since this is time-aware. In each of these folds, the training set contains data from 1992 to the year t, and the validation set contains data from the year t + 1 only. The earliest training set uses data through 1994 and advances by one year through the year 2013, producing a total of 20 folds. 



#### Table 3: AUC Results: Cross-validation results for wildfire prediction models

| train_end_year | valid_year | auc_glm | auc_xgb |
|--------------|-----------|----------|----------|
| 1994 | 1995 | 0.6701 | 0.8022 |
| 1995 | 1996 | 0.7229 | 0.7980 |
| 1996 | 1997 | 0.7930 | 0.8103 |
| 1997 | 1998 | 0.7593 | 0.8036 |
| 1998 | 1999 | 0.6683 | 0.6883 |
| 1999 | 2000 | 0.7276 | 0.7750 |
| 2000 | 2001 | 0.7559 | 0.8083 |
| 2001 | 2002 | 0.7232 | 0.7650 |
| 2002 | 2003 | 0.7285 | 0.7883 |
| 2003 | 2004 | 0.6615 | 0.7843 |
| 2004 | 2005 | 0.6648 | 0.7806 |
| 2005 | 2006 | 0.6645 | 0.6864 |
| 2006 | 2007 | 0.6467 | 0.7330 |
| 2007 | 2008 | 0.6149 | 0.7593 |
| 2008 | 2009 | 0.6775 | 0.8200 |
| 2009 | 2010 | 0.6548 | 0.6761 |
| 2010 | 2011 | 0.6609 | 0.6885 |
| 2011 | 2012 | 0.6351 | 0.7011 |
| 2012 | 2013 | 0.6985 | 0.7128 |
| 2013 | 2014 | 0.7384 | 0.7967 |

The table below shows AUC values from rolling-year cross-validation.  
Each row trains the model on all years up to `train_end_year` and evaluates on `valid_year`.  
We compare logistic regression (GLM) and gradient boosted trees (XGBoost).

#### Figure 4: Time-aware cross-validation diagnostic: AUC by validation year

<figure>
<img src="Fig12.png" width="700">
<figcaption>
The plot shows the average value of y_next by calendar month, aggregated over all grid cells and years in the dataset.

Across nearly all validation years, XGBoost achieves higher AUC than logistic regression, typically in the range of about 0.75–0.82, while logistic regression is usually between about 0.63–0.76. The performance does vary across years, which shows changes in wildfire activity and the difficulty of prediction in different periods. The consistent improvement from XGBoost suggests that there are some nonlinear aspects in the prediction model that the linear regression model was not able to capture.

</figcaption>
</figure>

#### Table 4: Rolling-year cross-validation performance for logistic regression and neural network models

| train_end_year | valid_year | auc_glm | auc_nn |
|--------------|-----------|----------|----------|
| 1994 | 1995 | 0.6566 | 0.6863 |
| 1995 | 1996 | 0.6756 | 0.7905 |
| 1996 | 1997 | 0.8241 | 0.7522 |
| 1997 | 1998 | 0.7882 | 0.7429 |
| 1998 | 1999 | 0.6518 | 0.7186 |
| 1999 | 2000 | 0.6758 | 0.7407 |
| 2000 | 2001 | 0.7448 | 0.7785 |
| 2001 | 2002 | 0.6760 | 0.7760 |
| 2002 | 2003 | 0.7275 | 0.8174 |
| 2003 | 2004 | 0.6722 | 0.7973 |
| 2004 | 2005 | 0.6400 | 0.7485 |
| 2005 | 2006 | 0.7049 | 0.7306 |
| 2006 | 2007 | 0.6111 | 0.6869 |
| 2007 | 2008 | 0.6103 | 0.7000 |
| 2008 | 2009 | 0.6769 | 0.7329 |
| 2009 | 2010 | 0.6786 | 0.7007 |
| 2010 | 2011 | 0.6778 | 0.6647 |
| 2011 | 2012 | 0.6797 | 0.7693 |
| 2012 | 2013 | 0.6895 | 0.7801 |
| 2013 | 2014 | 0.7333 | 0.7847 |

The table reports AUC values obtained using time-aware cross-validation, where each model is trained on all years up to the training cutoff and evaluated on the following year. This approach prevents information from the future from influencing model training.

The neural network generally achieves higher AUC than logistic regression in most validation years, often reaching values around 0.73–0.82 compared to about 0.61–0.82 for logistic regression. However, the improvement is not uniform across all years, and in some cases logistic regression performs similarly or slightly better. These results suggest that the neural network is able to capture additional nonlinear relationships in the predictors, but the simpler logistic regression model remains competitive and more stable across years.

#### Figure 5: Time-aware cross-validation diagnostic: AUC by validation year for logistic regression and neural network models

<figure>
<img src="Fig13.png" width="700">
<figcaption>
The plot shows AUC values obtained from rolling time-aware cross-validation. This evaluation is different in the sense that it respects the temporal ordering of the data and prevents data leakage.

Across most of the validation years, we see that the neural network achieves higher AUC than logistic regression, with values between 0.70 and 0.82. Logistic regression's range is between 0.61 and 0.74. The improvement is the most noticeable in the early 2000s and after 2011, when the neural network captures patterns that the linear model was unable to do. The use of a neural network here is very important because the features of a neural network are to understand the underlying nonlinear complexity of a dataset.

However, performance does vary quite a bit from year to year; this is reflective of changes in wildfire activity and the difficulty of predicting rare events. In a few years, logistic regression performs similarly or slightly better, suggesting that the additional flexibility of the neural network does not always mean better performance.

</figcaption>
</figure>

#### Table 5: Comparison of the Models Summary

| Model | Features | Mean AUC (20 folds) | Relative Gain vs. Logistic |
|--------|----------|---------------------|-----------------------------|
| Logistic Regression (full) | 12 | 0.690 | — |
| XGBoost | 12 | 0.755 | +9.46% |
| Logistic Regression (reduced) | 5 | 0.699 | — |
| Neural Network | 5 | 0.740 | +5.86% |

The full logistic regression model with 12 predictors has a mean AUC of 0.690 and is the baseline.  
XGBoost provides the best performance with a mean AUC of 0.755.
Using a reduced feature set, logistic regression achieves similar performance (0.699), while the neural network improves to 0.740, showing that nonlinear models can capture additional structure in the data even with fewer predictors.  

 ## 6. Conclusion & Limitations

This analysis therefore demonstrates that wildfires can be predicted a month in advance using the previous data sets from Kaggle. The key findings from this analysis are that XGBoost consistently has outperformed logistic regression across all of the 20 folds, reaching a mean AUC of 0.755 versus 0.690.  This performance shows that there is a nonlinear component to the data set, and that logistic regression was too simple to capture this. XGBoost was able to, showing that it is the more robust model relative to the two. 

The neural network achieved immediate performance with a mean AUC of -0.740 on the five features, which were the reduced features, confirming that models that are non-linear are more advantageous even if the features are reduced for this dataset because of the temporal aspect. Of all of the predictors, it seems that the spatial predictors that include longitude, latitude, and state are the ones that carry the most impact, especially shown in the California-specific random forest (see below in Additional Insights). 

Seasonality was incredibly important to being coded as harmonic using the sine and cosine month features in order to capture that summer has the peak in terms of fire risk and also provides a complementary signal to the lagged fire history features. The panel structure reflects how rare and spatially concentrated notable fires can be in the United States. AUC is the appropriate metric here because it measures the discrimination across the full range of all of the classification thresholds. 

Some limitations include:
- The data set was not the complete one that was taken from Kaggle; it was only a subset of about 100,000 records.
- We also did not study each of the variables that were given in this data set, including temperature, humidity, and wind, which are known to be very strong factors in wildfires.
- Furthermore, we could improve this model in order to help the prediction value by including other data sets that are focused more on temperature and more spatial-temporal regions, along with weather station data or maybe even spatial grids that are smaller and finer.

## 7. Additional Insights (California Study)

Because my hometown is very prone to wildfires back in Southern California, I took this analysis one step further and conducted a supplementary random forest analysis exclusively on California wildfire records, which was about 10,089 observations from the full dataset. The binary outcome variable of big_fire was defined to be FIRE_SIZE > 1 acre. The random forest was fit with 200 trees, and two variables were tried at each of the splits. The following predictors are tried:
- latitude-longitude
- month-year
- stat-cause_descr (statutory cause description)
- owner-discr (land ownership type)

The model had a mean of squared residuals of 0.1556 and explained about 4.3% of the variance, which is low in absolute terms.  The variable importance was measured by %INCMSE.

#### Why Random Forest?

For the California-focused analysis, I chose a Random Forest model because the California subset had fewer observations compared to the much larger United States dataset. This made it less suitable for models such as XGBoost or neural networks that require larger data sets in order to perform to their best ability. 

Random Forest also provides a strong nonlinear baseline that can capture a lot of spatial, seasonal, and historic fire variables without requiring extensive pre-processing and hyperparameters compared to the other models that were used earlier. In addition, Random Forest also allows for straightforward estimation of how important a variable is, which makes it a good choice for this additional exploratory analysis.

#### Table 6: Variable importance from Random Forest model (California subset)

| Variable | %IncMSE | IncNodePurity |
|----------|---------|---------------|
| LATITUDE | 36.28682 | 390.73453 |
| LONGITUDE | 32.77287 | 378.04722 |
| month | 16.51513 | 145.60481 |
| year | 26.89806 | 194.84292 |
| STAT_CAUSE_DESCR | 20.89182 | 98.24956 |
| OWNER_DESCR | 34.85403 | 79.06757 |

From this table, we can see that the geographic coordinates of latitude and longitude, as well as land ownership, are the three most important predictors indicated by this random forest model. Year also does contribute substantially, which captures a trend towards larger fires over the period. We see that statutory cause and month are not as important as these first three, but they do still give us some information.

Our main inferences about which factors would contribute to a wildfire, including where it is in the United States, the vegetation, the dryness, the terrain, and the management, are going to be more predictive of the size of a fire and whether or not it is therefore notable. 